# XGBoost Training — SPY Direction with Optuna Hyperparameter Tuning
Optimises AUC over 50 Optuna trials, then evaluates with SHAP feature importance.

In [ ]:
!pip install -q xgboost optuna yfinance shap pandas numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import shap
import matplotlib.pyplot as plt
import warnings
import joblib
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SYMBOL  = 'SPY'
N_TRIALS = 50
print(f'XGBoost + Optuna | symbol={SYMBOL} | trials={N_TRIALS}')

In [ ]:
import yfinance as yf

df = yf.download(SYMBOL, start='2015-01-01', interval='1d', progress=False)
df = df.dropna()
print(f'Downloaded {len(df)} daily bars for {SYMBOL}')

close = df['Close'].squeeze()
high  = df['High'].squeeze()
low   = df['Low'].squeeze()
vol   = df['Volume'].squeeze()

# Returns
df['ret_1d']  = close.pct_change()
df['ret_5d']  = close.pct_change(5)
df['ret_20d'] = close.pct_change(20)

# Momentum
df['mom_10'] = close / close.shift(10) - 1
df['mom_20'] = close / close.shift(20) - 1

# RSI
def rsi(s, n=14):
    d = s.diff(); g = d.clip(lower=0); l = -d.clip(upper=0)
    return 100 - 100 / (1 + g.ewm(com=n-1, min_periods=n).mean() /
                             (l.ewm(com=n-1, min_periods=n).mean() + 1e-9))

df['rsi_14'] = rsi(close)

# MACD
df['macd'] = close.ewm(span=12).mean() - close.ewm(span=26).mean()
df['macd_signal'] = df['macd'].ewm(span=9).mean()

# Bollinger Bands
bb_mid = close.rolling(20).mean()
bb_std = close.rolling(20).std()
df['bb_pct'] = (close - bb_mid) / (bb_std * 2 + 1e-9)
df['bb_width'] = bb_std * 2 / (bb_mid + 1e-9)

# ATR
tr = pd.concat([
    high - low,
    (high - close.shift()).abs(),
    (low  - close.shift()).abs()
], axis=1).max(axis=1)
df['atr_14'] = tr.rolling(14).mean() / (close + 1e-9)

# Volume
df['vol_ma5']  = vol / (vol.rolling(5).mean() + 1e-9)
df['vol_ma20'] = vol / (vol.rolling(20).mean() + 1e-9)

# VIX proxy (realised vol)
df['realised_vol_20'] = df['ret_1d'].rolling(20).std() * np.sqrt(252)

# Target: next-day up (1) or down (0)
df['target'] = (close.shift(-1) > close).astype(int)

FEATURES = ['ret_1d', 'ret_5d', 'ret_20d', 'mom_10', 'mom_20',
            'rsi_14', 'macd', 'macd_signal', 'bb_pct', 'bb_width',
            'atr_14', 'vol_ma5', 'vol_ma20', 'realised_vol_20']

df = df[FEATURES + ['target']].dropna()
print(f'Features: {len(FEATURES)} | Samples: {len(df)}')

In [ ]:
n = len(df)
train_end = int(n * 0.75)
val_end   = int(n * 0.88)

X = df[FEATURES].values
y = df['target'].values

X_train, y_train = X[:train_end],    y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],      y[val_end:]

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

def objective(trial: optuna.Trial) -> float:
    params = {
        'objective':        'binary:logistic',
        'eval_metric':      'auc',
        'tree_method':      'hist',
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 9),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
        'verbosity':        0,
        'use_label_encoder': False,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=20,
        verbose=False,
    )
    proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, proba)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)

print(f'\nBest trial: AUC = {study.best_value:.4f}')
print('Best params:', study.best_params)

In [ ]:
best_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'verbosity': 0,
    'use_label_encoder': False,
    **study.best_params,
}
best_model = xgb.XGBClassifier(**best_params)
best_model.fit(
    np.vstack([X_train, X_val]),
    np.concatenate([y_train, y_val]),
    verbose=False,
)

# Evaluate on test set
proba_test = best_model.predict_proba(X_test)[:, 1]
preds_test = best_model.predict(X_test)
test_auc   = roc_auc_score(y_test, proba_test)
test_acc   = accuracy_score(y_test, preds_test)
print(f'Test AUC:      {test_auc:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
# SHAP feature importance
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test[:500])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
fi_df = pd.DataFrame({'feature': FEATURES, 'importance': mean_abs_shap})
fi_df = fi_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi_df['feature'], fi_df['importance'], color='#2979ff')
ax.set_title('SHAP Feature Importance (mean |SHAP|)')
ax.set_xlabel('Mean |SHAP value|')
ax.set_facecolor('#1e2433')
fig.patch.set_facecolor('#131722')
ax.tick_params(colors='white')
ax.title.set_color('white')
ax.xaxis.label.set_color('white')
plt.tight_layout()
plt.savefig('xgboost_shap.png', dpi=100, bbox_inches='tight')
plt.show()
print('SHAP plot saved to xgboost_shap.png')

In [ ]:
import os

save_path = 'xgboost_spy.json'
best_model.save_model(save_path)
print(f'Model saved to {save_path} ({os.path.getsize(save_path)/1024:.1f} KB)')

# Save scaler + feature list for inference
joblib.dump({'features': FEATURES}, 'xgboost_spy_meta.pkl')
print('Metadata saved to xgboost_spy_meta.pkl')